# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.

Solution: No. One hundred successful deliveries show that the robot can perform the task under the particular conditions represented in those episodes but they do not establish that it can reliably perform the task across arbitrary home environments. The robot's competence depends on the relationship between its control program, the manipulation problem, body and sensors, and the world in which it operates.
A wider competence claim requires testing across different mugs, fill levels, routes, human positions, and environmental conditions. Failures and changed conditions must also be recorded and evaluated.

Confidence: 4/5     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.

Solution: If the mug is nearly full instead of half-full, the robot may spill coffee while carrying it along the same route. The increased liquid volume makes the mug more sensitive to acceleration, deceleration, and tilting. The untested assumption in the original run was that the same trajectory would remain safe for different fill levels. The robot should update its representation of the mug's fill level, predict the consequences of the changed condition using its Semantic Digital Twin, and check whether the proposed motion satisfies the safety constraints before execution.
     
3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.

Solution: The robot should ask Eva whether the white mug is an acceptable substitute before proceeding. Evidence is required regarding Eva's explicit confirmation that she accepts the white mug. The robot cannot infer that the available mug satisfies the intended goal merely because it is physically present. This is a grounding problem as the system must connect the natural-language request to the correct object, constraints, and desired outcome in the current situation. Until acceptance is established, the proposed action does not satisfy the grounding criterion of the Warrant Gate. The robot should therefore refrain from committing to the action.

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.

Solution: They are as follows:
**T1 – Explain** --->	Why did the robot deliver the half-full mug without spilling? Was it due to the slow turning motion or the lower fill level?
**T2 – Predict** --->	Will the same turning trajectory cause the nearly full mug to spill?
**T3 – Design**	---> Would using a tray or reducing the tilt rate improve the stability of the nearly full mug during transportation?
**T4 – Bound**	---> For which mug types, fill levels, routes, and environmental conditions can the robot reliably deliver coffee without spilling?
**T5 – Check** ---> Does the episodic audit record contain Eva's acceptance, the selected motion parameters, sensor observations, and the actual delivery outcome needed to evaluate success?
These five questions form a cycle of explaining an episode, predicting changed cases, designing interventions, bounding the scope of competence, and checking the evidence.

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.

Solution: To handle Eva's request of bringing her coffee safely, the robot requires several interconnected functional modules:

1. **Interpretation:** Understand Eva's request and identify the goal of delivering coffee to her.
2. **Perception:** Use sensor data to identify objects, locate the mug, detect obstacles, and estimate Eva's position.
3. **Proposer (System 1):** Generate a possible sequence of actions to achieve the requested task.
4. **Semantic Digital Twin (System 2):** Represent the environment, objects, and their properties, and predict the possible outcomes of actions before execution.
5. **Warrant Gate:** Check whether the proposed action is grounded in evidence, feasible, safe, monitorable, repairable, and recordable before allowing execution.
6. **Executive:** Select and coordinate the approved actions and manage the execution process.
7. **Execution Engine (Acting):** Translate the selected actions into robot movements and carry out the task while monitoring changes in the environment.
8. **Episodic Audit Trail (Recording):** Store the observations, decisions, actions, and outcomes so that the execution can be reviewed and verified later.

These modules work together as an integrated system. The robot first interprets the request and perceives the environment. The Proposer generates a plan, while the Semantic Digital Twin predicts its consequences. The Warrant Gate evaluates the plan before the Executive authorises its execution. Finally, the Execution Engine performs the actions, and the Episodic Audit Trail records the entire process. If the situation changes during execution, the system can update its perception and re-evaluate the plan.
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

Solution: If Eva suddenly stands up while the robot is carrying coffee, the robot should pause or stop its current motion safely. It must update its perception and internal representation of Eva's position and the surrounding situation. The robot should then use its Semantic Digital Twin to predict the consequences of possible actions and generate a revised plan. Before continuing, the revised plan must pass the Warrant Gate to ensure safety and feasibility. The robot should also record the unexpected event and its response in the Episodic Audit Trail.


## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [ ]:

#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time
import math


@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation"""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float


@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters"""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"
    contact_depth_m: float = 0.015


class Proposer:
    """System 1: Generates candidate execution parameters."""

    def __init__(self):
        self.velocity_mean = 0.5
        self.velocity_std = 0.15
        self.tilt_mean = 0.1
        self.tilt_std = 0.03

    def propose(self, goal: GoalSpecification, velocity_override: Optional[float] = None) -> ParameterPacket:
        velocity = random.gauss(self.velocity_mean, self.velocity_std)

        if velocity_override is not None:
            velocity = velocity_override

        velocity = max(0.05, min(velocity, goal.max_nav_velocity_mps))
        tilt_rate = max(0.01, random.gauss(self.tilt_mean, self.tilt_std))

        return ParameterPacket(
            grasp_pose=(0.45, 0.12, 0.30),
            max_velocity_mps=velocity,
            tilt_rate_radps=tilt_rate,
            source_generator="gaussian_sampler"
        )


@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str


@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True

    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"

    # Task 4: Battery information
    battery_capacity_wh: float = 100.0
    battery_level_wh: float = 100.0


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context
    into precise physical goal parameters (GoalSpecification).
    """

    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")

        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})

        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor

        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0

        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )

        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations"""

    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)

        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations"""

    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps"""

    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:

        stiffness_coefficient = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)

        # Task 2.1: Increase stiffness exponentially when contact depth exceeds 0.02 m
        if params.contact_depth_m > 0.02:
            stiffness_coefficient *= math.exp(
                1.8 * (params.contact_depth_m - 0.02)
            )

        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)

        # Task 3.1: Viscosity decreases by 1% per degree above 20°C
        effective_viscosity = self.belief.fluid_viscosity_pas * (
            1.0 - (self.belief.ambient_temperature_c - 20.0) * 0.01
        )
        effective_viscosity = max(0.01, effective_viscosity)

        radius = 0.05
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity
        predicted_wave = wave_height_m * 1000.0

        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps)
        }


class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification"""

    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)

        # Task 4.1: Estimate battery drain for the proposed trajectory
        estimated_battery_drain_wh = (
            params.max_velocity_mps * 10.0
            + params.tilt_rate_radps * 5.0
        )

        energetically_viable = (
            estimated_battery_drain_wh < 0.15 * belief.battery_capacity_wh
            and estimated_battery_drain_wh <= belief.battery_level_wh
        )

        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n)
                    and (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm)
                    and (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            "energetically_viable": energetically_viable
        }

        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)

        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks,
            "estimated_battery_drain_wh": round(estimated_battery_drain_wh, 2)
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail"""

    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)

        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)

        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"

        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome

    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState,
                     proposal: ParameterPacket, warrant_result: Dict[str, Any],
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)



# Task 5.1: Analyze the episodic audit trail
def audit_analyzer(audit_trail: List[Dict[str, Any]]) -> None:
    rejected_proposals = 0
    approved_executions = 0
    failed_criteria = []
    force_variances = []

    for record in audit_trail:
        committed = record["committed"]
        warrant = committed["warrant_verdict"]
        outcome = record["execution_outcome"]

        if not committed["action_approved"]:
            rejected_proposals += 1

            if warrant["failed_criterion"] is not None:
                failed_criteria.append(warrant["failed_criterion"])

        if committed["action_approved"] and outcome is not None:
            approved_executions += 1

            predicted_force = warrant["predictions"]["predicted_force_n"]
            observed_force = outcome["actual_force_n"]

            # Calculate squared prediction error
            force_variance = (predicted_force - observed_force) ** 2
            force_variances.append(force_variance)

    most_frequent_failed_criterion = (
        max(set(failed_criteria), key=failed_criteria.count)
        if failed_criteria else "None"
    )

    # Calculate average squared force prediction error (MSE)
    average_force_variance = (
        sum(force_variances) / len(force_variances)
        if force_variances else 0.0
    )

    print("\n=== AUDIT ANALYZER ===")
    print(f"Rejected proposals: {rejected_proposals}")
    print(f"Approved executions: {approved_executions}")
    print(f"Most frequent failed criterion: {most_frequent_failed_criterion}")
    print(f"Average force variance (MSE): {average_force_variance:.4f} N²")


# --- Execution Demonstration ---
if __name__ == "__main__":

    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    proposer = Proposer()

    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5,
        true_friction=0.45,
        eva_pos="living_room",
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")

    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)

    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8,
        tilt_rate_radps=0.2
    )
    print(f"Proposer suggests: v_max = {proposal_v1.max_velocity_mps} m/s")

    print("\n=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===")
    warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
    print(f"SDT Predictions: Nav-Vel = {warrant_v1['predictions']['predicted_nav_velocity']} m/s")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")
    print(f"Gate Verdict    : REJECTED (Failed on: {warrant_v1['failed_criterion']})")

    executive.record_trace(
        raw_command, goal, belief_state, proposal_v1,
        warrant_v1, raw_observation, committed=False
    )

    
    # Task 2.2: Proposer generates an aggressive candidate
    print("\n=== TASK 2.2: Aggressive Proposal Test ===")
    aggressive_proposal = proposer.propose(goal, velocity_override=0.35)
    aggressive_proposal.contact_depth_m = 0.025
    aggressive_proposal.tilt_rate_radps = 0.20

    aggressive_warrant = warrant_gate.validate(
        aggressive_proposal, goal, belief_state
    )

    print(f"Proposed velocity: {aggressive_proposal.max_velocity_mps} m/s")
    print(f"Proposed tilt rate: {aggressive_proposal.tilt_rate_radps} rad/s")
    print(f"Predicted force: {aggressive_warrant['predictions']['predicted_force_n']} N")
    print(f"Predicted wave height: {aggressive_warrant['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant checks: {aggressive_warrant['warrant_checks']}")
    print(f"Gate verdict: {'APPROVED' if aggressive_warrant['passed'] else 'REJECTED'}")

    if not aggressive_warrant["passed"]:
        executive.record_trace(
            raw_command, goal, belief_state, aggressive_proposal,
            aggressive_warrant, raw_observation, committed=False
        )
        print("Warrant Gate intercepted the proposal before physical execution.")


    # Task 3.2: Test prediction at 28°C and tilt rate 0.15 rad/s
    print("\n=== TASK 3.2: Temperature and Slosh Test ===")
    original_temperature = belief_state.ambient_temperature_c
    belief_state.ambient_temperature_c = 28.0

    temperature_test_proposal = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.2,
        tilt_rate_radps=0.15
    )

    temperature_test_warrant = warrant_gate.validate(
        temperature_test_proposal, goal, belief_state
    )

    effective_viscosity = 1.2 * (1.0 - (28.0 - 20.0) * 0.01)
    predicted_wave_without_error = (
        0.15 * 0.05 / effective_viscosity
    ) * 1000.0

    print(f"Effective viscosity at 28°C: {effective_viscosity:.3f} Pa·s")
    print(f"Predicted wave height (before epistemic error): {predicted_wave_without_error:.2f} mm")
    print(f"SDT predicted wave height: {temperature_test_warrant['predictions']['predicted_wave_mm']} mm")
    print(f"Maximum allowable wave height: {goal.max_slosh_wave_mm} mm")
    print(f"Safe: {temperature_test_warrant['warrant_checks']['safe']}")

    belief_state.ambient_temperature_c = original_temperature

    # Task 4.2: Low battery should prevent coffee delivery and request recharge
    print("\n=== TASK 4.2: Low Battery Test ===")
    original_battery_level = belief_state.battery_level_wh
    belief_state.battery_level_wh = 1.0

    low_battery_proposal = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.2,
        tilt_rate_radps=0.01
    )

    low_battery_warrant = warrant_gate.validate(
        low_battery_proposal, goal, belief_state
    )

    print(f"Estimated battery drain: {low_battery_warrant['estimated_battery_drain_wh']} Wh")
    print(f"Battery remaining: {belief_state.battery_level_wh} Wh")
    print(f"Energetically viable: {low_battery_warrant['warrant_checks']['energetically_viable']}")

    if not low_battery_warrant["warrant_checks"]["energetically_viable"]:
        print("[EXECUTIVE] Battery insufficient. Requesting recharge.")
        print("[EXECUTIVE] Coffee delivery postponed; no physical execution.")

        executive.record_trace(
            raw_command, goal, belief_state, low_battery_proposal,
            low_battery_warrant, raw_observation, committed=False
        )

    belief_state.battery_level_wh = original_battery_level

    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")
    proposal_v2 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.45,
        tilt_rate_radps=0.02
    )

    warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
    print(f"SDT Predictions: Force = {warrant_v2['predictions']['predicted_force_n']} N, Wave = {warrant_v2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")
    print(f"Gate Verdict    : {'APPROVED' if warrant_v2['passed'] else 'REJECTED'}")

    print("\n=== STEP 6: Physical Execution & Epistemic Audit Trail ===")
    if warrant_v2["passed"]:
        final_outcome = executive.execute_action(proposal_v2, goal, warrant_v2)
        executive.record_trace(
            raw_command, goal, belief_state, proposal_v2,
            warrant_v2, raw_observation, committed=True,
            execution_outcome=final_outcome
        )
        print(f"Execution Outcome: {final_outcome}")
    else:
        executive.record_trace(
            raw_command, goal, belief_state, proposal_v2,
            warrant_v2, raw_observation, committed=False
        )
        print("Warrant Gate rejected the proposal. No physical execution.")

    # Task 5.1: Analyze all entries in the episodic audit trail
    audit_analyzer(executive.audit_trail)

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")


=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793709783.653667, measured_temperature_c=21.67, measured_friction=0.43, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.67°C, Friction=0.43

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: System 1 Proposer Generates Parameters ===
Proposer suggests: v_max = 0.8 m/s

=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===
SDT Predictions: Nav-Vel = 0.72 m/s
Warrant Checks  : {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True, 'energetically_viable': True}
Gate Verdict    : REJECTED (Failed on: safe)

=== TASK 2.2: Aggressive Proposal Test ===
Proposed veloci

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.

**Solution:** The classes in the Python script correspond to the main functional blocks of the integration diagram.

- **Interpretation:** `InterpretationSystem` converts Eva's request into a `GoalSpecification` containing the target object, destination and task constraints.
- **Perception:** `SensingSystem` collects sensor observations, while `PerceptionSystem` updates the robot's internal `BeliefState` based on these observations.
- **System 1 (Proposer):** `Proposer` generates possible execution parameters, such as grasp pose, velocity and tilt rate.
- **System 2 (Semantic Digital Twin):** `SemanticDigitalTwin.predict_properties()` predicts the physical consequences of the proposed parameters.
- **Warrant Gate:** `WarrantGate.validate()` checks whether the proposal satisfies the required conditions, including grounding, feasibility, safety, monitoring, repair and recording.
- **Executive:** The main execution workflow coordinates the systems and decides whether to execute the proposal or request another one.
- **Acting:** `ExecutionEngine.execute_action()` performs the approved action and simulates its physical outcome.
- **Episodic Audit Trail:** `ExecutionEngine.record_trace()` records the request, belief, proposal, warrant decision and execution outcome.

During a decision cycle, Eva's request is first interpreted into a goal. The robot then senses its environment and updates its belief. The Proposer generates a candidate action, and the Semantic Digital Twin predicts its possible consequences. The Warrant Gate evaluates the proposal against the task constraints. If the proposal fails, the Executive can request another proposal. Otherwise, the Execution Engine carries out the action and records the outcome for later analysis.


* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.

**Solution:** Made changes to the code. Justification: I chose a Gaussian distribution for the Proposer because it generates parameters around a typical operating value while still allowing some variation between proposals. This allows the system to explore different possible actions without sampling completely random values each time. The velocity is also limited by the maximum navigation velocity specified in the goal. The generated parameters are only candidate actions, so they still need to be evaluated by the Semantic Digital Twin and Warrant Gate before execution.

  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.

**Solution:** Changes made to the code 

* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.


**Solution:** Changes made to the code and calculation are as follows.


The proposer generates an aggressive approach velocity of 0.35 m/s and a contact depth of 0.025 m. The surface friction coefficient is 0.46 and the ambient temperature is 21.67°C.

The stiffness coefficient is calculated as:

$$
k = 12.5(1 + 0.46 \times 0.1) = 13.075
$$

Since the contact depth exceeds 0.02 m, the bottoming effect is applied:

$$
k' = 13.075 \times e^{1.8(0.025-0.02)}
$$

$$
k' \approx 13.193
$$

The predicted force before epistemic noise is:

$$
F = k' \times (0.35)^{1.5} \approx 2.73 \text{ N}
$$

The effective viscosity is:

$$
\mu_{\text{eff}} = 1.2(1-0.01(21.67-20))
$$

$$
\mu_{\text{eff}} \approx 1.17996 \text{ Pa}\cdot\text{s}
$$

The predicted wave height before epistemic noise is:

$$
h = \frac{0.20 \times 0.05}{1.17996} \times 1000
\approx 8.47 \text{ mm}
$$

The predicted force is below the maximum allowable force of 4.5 N. However, the predicted wave height exceeds the maximum allowable wave height of 1.5 mm. Therefore, the `safe` criterion fails, and the Warrant Gate rejects the proposal before physical execution.


### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).

**Solution:** Made changes to the code

* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

**Solution:**

At an ambient temperature of 28°C, the effective viscosity is calculated using the linear temperature-dependent model, which decreases viscosity by 1% for every degree Celsius above 20°C.

$$
\mu_{\text{eff}} = 1.2(1-0.01(28-20))
$$

$$
\mu_{\text{eff}} = 1.104 \text{ Pa}\cdot\text{s}
$$

For a tilt rate of 0.15 rad/s, the predicted wave height is:

$$
h = \frac{0.15 \times 0.05}{1.104} \times 1000
$$

$$
h \approx 6.79 \text{ mm}
$$

Since the predicted wave height is greater than the maximum allowable wave height:

$$
6.79 \text{ mm} > 1.5 \text{ mm}
$$

the `safe` criterion fails. Therefore, the Warrant Gate rejects the proposal before execution to prevent an unsafe action.


### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.

**Solution:**

An `energetically_viable` criterion was added to the Warrant Gate to ensure that a proposed action does not consume an excessive amount of energy.

The estimated battery drain is calculated based on the proposed navigation velocity and tilt rate:

$$
E_{\text{drain}} = 10v + 5\omega
$$

The criterion checks whether the estimated battery drain is less than 15% of the total battery capacity:

$$
E_{\text{drain}} < 0.15E_{\text{capacity}}
$$

It also verifies that the estimated drain does not exceed the currently available battery level:

$$
E_{\text{drain}} \leq E_{\text{remaining}}
$$

If either condition is not satisfied, the `energetically_viable` criterion fails and the Warrant Gate rejects the proposal. This ensures that the robot considers both its total energy capacity and its remaining battery charge before executing an action.


* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

**Solution:** The `BeliefState` was extended with `battery_capacity_wh` and `battery_level_wh` to represent the total battery capacity and the remaining charge. When the battery level is insufficient to perform the proposed action, the `energetically_viable` criterion fails. The Warrant Gate rejects the proposed delivery action, and the Executive redirects the workflow to recharge instead of proceeding with the delivery. This ensures that energy constraints are explicitly considered during action selection and that the robot does not attempt to execute a task when its battery is insufficient.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.

**Solution:** The `audit_analyzer(audit_trail)` function analyzes the episodic audit trail to evaluate the performance and reliability of the robotic system.

It performs the following operations:

1. Counts the number of rejected proposals.
2. Counts the number of approved proposals that proceeded to execution.
3. Identifies the most frequently failed criterion among rejected proposals.
4. Calculates the average squared difference between predicted and observed forces for approved executions.

The average force prediction variance is calculated as:

$$
\text{Average Squared Error} = \frac{1}{n}\sum_{i=1}^{n}(F_{\text{predicted},i}-F_{\text{observed},i})^2
$$

This metric is the Mean Squared Error (MSE). It measures the magnitude of prediction errors and helps evaluate the discrepancy between the Semantic Digital Twin's predictions and actual physical outcomes, thereby quantifying the sim-to-real gap.



* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

**Solution:** Pure data-driven imitation learning learns a mapping from observed inputs to actions by reproducing patterns in training data. However, it does not necessarily represent the underlying reasons for an action, the agent's internal beliefs, or whether an action was considered safe before execution. The 4 separate records in the episodic audit trail address this epistemological gap:

1. **Asked:** Records the original natural-language instruction and the interpreted goal. This preserves what the user actually requested.
2. **Belief:** Records the robot's internal belief about the environment, including temperature, friction, Eva's position, and room busyness. This distinguishes internal knowledge from external reality.
3. **Committed:** Records the proposed parameters and Warrant Gate verdict, including whether the action was approved or rejected. This makes the reasoning behind action selection traceable.
4. **Observed:** Records the actual sensor measurements obtained from the environment. These measurements may differ from the robot's internal beliefs due to sensor noise or uncertainty.

Separating these records allows the system to distinguish between what was requested, what it believed, what it decided, and what actually happened. This supports explainability, debugging, safety verification, and learning from failures. Therefore, unlike pure imitation learning, the architecture maintains an explicit epistemic audit trail that makes its decisions interpretable and accountable.


## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---
**Solution:** The key functional parts of a robot and their roles in the embodied cognitive architecture are:

1. **Links:** These are the rigid structural elements of the robot that connect joints and form its body. They provide the physical structure through which motion and forces are transmitted.

2. **Joints:** They connect links and allow relative movement between them. They define the robot's degrees of freedom and enable the execution of planned movements.

3. **Actuators:** These are the components that generate physical motion by converting energy into mechanical movement. They execute the commands produced by the control and decision-making components of the architecture.

4. **Sensors:** They are devices that measure information about the robot and its surroundings. Their measurements are used by the Sensing and Perception systems to update the robot's internal Belief State.

5. **Effectors:** The components that directly interact with the environment to perform tasks are called Effectors. For example, a gripper grasps and holds Eva's coffee mug.

6. **Base:** It is the supporting platform of the robot, which enables locomotion and provides stability during navigation and manipulation.

7. **Torso:** It is the central body structure that supports and connects other components, such as the arms, head, and base. It provides structural support and may enable additional movement.

8. **Head:** It usually carries sensors such as cameras and other perception devices. It helps the robot observe and perceive its surroundings.

9. **Wheels:** They enable locomotion through rotation and are typically suitable for efficient movement on relatively flat surfaces.

10. **Legs:** They enable locomotion over uneven terrain and can provide additional mobility and adaptability through coordinated joint movements.

11. **Proprioception:** It is the robot's ability to sense its own internal state, such as joint positions, velocities, and forces. It supports feedback control and accurate execution of movements.

12. **Exteroception:** The sensing of external environmental properties through sensors such as cameras, lidar, and proximity sensors is known as Exteroception. It supports perception, navigation, and obstacle avoidance.

13. **Interoception:** The sensing of internal physiological or operational states, such as battery level, motor temperature, and energy consumption is known as Interoception. It helps maintain the robot's operational condition and supports the energetic viability criterion of the Warrant Gate.

All these components work together in the embodied cognitive architecture. The Proposer generates candidate actions, the Semantic Digital Twin predicts their consequences, and the Warrant Gate evaluates their feasibility and safety. Once approved, the Execution Engine uses actuators and effectors to perform the action. Sensors provide feedback to the Perception system, which updates the Belief State and enables the robot to monitor and adapt its behavior.


# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 